# CA2 Part A: Customer Segmentation (Clustering)

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"

### 0) Setup + Load Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture

from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

df = pd.read_csv("../data/CA2-Customer-Data.csv")
df.head()

In [ ]:
print(df.shape)
df.info()

In [ ]:
df.describe(include="all")

In [ ]:
df["Gender"].value_counts()

In [ ]:
df["Gender"].value_counts(normalize=True) * 100

---
### Initial Data Understanding

The dataset contains 200 customers with no missing values across all features.
Gender distribution is slightly skewed towards females (56%), but both genders are well represented.

Age ranges from 18 to 70, while income and spending show much wider ranges.
Since clustering is distance-based, income and spending would dominate the distance calculations if left unscaled.
Therefore, feature scaling is required before applying clustering algorithms.

---

### 1) EDA

**1.1 Check missing values + duplicates**

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum()

**1.2 Gender balance**

In [ ]:
gender_counts = df["Gender"].value_counts()
gender_counts

In [ ]:
plt.figure()
plt.bar(gender_counts.index, gender_counts.values)
plt.title("Gender Balance")
plt.xlabel("Gender")
plt.ylabel("Count")
plt.show()

**Gender Balance**

The dataset shows a slight imbalance, with more female (56%) than male (44%) customers.
Both genders are sufficiently represented, so gender is suitable for post-cluster profiling rather than clustering.

**1.3 Distributions (Age, Income, Spending)**

In [ ]:
num_cols = ["Age", "Income (k$)", "How Much They Spend"]

plt.figure(figsize=(12,4))
for i, c in enumerate(num_cols, 1):
    plt.subplot(1, 3, i)
    plt.hist(df[c], bins=15)
    plt.title(f"Distribution: {c}")
plt.tight_layout()
plt.show()

**Distribution: Age**

Most customers fall between their late 20s and late 40s, with fewer customers at the extreme younger and older ages.
The age distribution suggests multiple age groups that may differ in spending behaviour.

**Distribution: Income**

Income is widely spread, ranging from low to high values, indicating strong variation in customer purchasing power.
A few high-income customers appear potential outliers.

**Distribution: How Much They Spend**

Spending values span almost the full range, with visible concentration around mid-level spending.
This suggests the presence of distinct spending segments rather than a single uniform group.

**1.4 Pairwise relationships**

In [ ]:
plt.figure()
plt.scatter(df["Income (k$)"], df["How Much They Spend"])
plt.title("Income vs Spending")
plt.xlabel("Income (k$)")
plt.ylabel("How Much They Spend")
plt.show()

**Income vs Spending**

Customers with similar income levels exhibit very different spending behaviours, indicating income alone does not determine spending.
The visible grouping patterns suggest clustering may help uncover meaningful customer segments.

In [ ]:
plt.figure()
plt.scatter(df["Age"], df["How Much They Spend"])
plt.title("Age vs Spending")
plt.xlabel("Age")
plt.ylabel("How Much They spend")
plt.show()

**Age vs Spending**

Younger customers tend to show more variability in spending, including both high and low spenders.
Olders customers appear more concentrated around moderate spending levels.

**1.5 Outliers (boxplots)**

In [ ]:
plt.figure()
plt.boxplot([df["Age"], df["Income (k$)"], df["How Much They Spend"]], tick_labels=["Age", "Income", "Spending"])
plt.title("Boxplots for Outlier Check")
plt.show()

**Boxplot for Outlier Check**

Income shows the widest spread and contains a high-value outlier, while age is more tightly distributed,
Due to these differing scales, feature scaling is neccessary before applying distance-based clustering.

---

### 2)Preprocessing + Feature Engineering

#### 2.1 Encode Gender (for profiling)

Gender is encoded for analysis purposes but excluded from clustering because it is categorical and binary.
Including it in distance-based clustering may cause clusters to form primarily by gender rather than customer behaviour.

In [ ]:
df["Gender_num"] = df["Gender"].map({"Female": 0, "Male": 1})
df[["Gender", "Gender_num"]].head()

#### 2.2 Select features + scale

Clustering algorithms rely on distance calculations.
Since income and spending have much larger numerical ranges than age, StandardScaler is applied to ensure all features contribute equally to the clustering process.

In [ ]:
X = df[["Age", "Income (k$)", "How Much They Spend"]].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

#### 2.3 Train/Test split

Although clustering is unsupervised, the dataset is split into training and test sets to evaluate cluster stability and structure on unseen data.

In [ ]:
X_train, X_test = train_test_split(X_scaled, test_size=0.2, random_state=42)

---
### 3) Model Building 

Beside K-Means, I considered other clustering algorithms from scikit-learn.

1) Agglomerative (Hierarchical) Clustering was chosen because it does not rely on centroid updates and can capture clusters that are not strictly spherical. This provides a strong comparison to K-Means for customer segmenattion.
   
2) Gaussian Mixture Model (GMM) was chosen because it performs probabilistic clustering, allowing customers to belong to clusters with different likelihoods. It can also model elliptical cluster shapes, which may better reflect overlapping customer behaviours in income and spending.

These models provide alternative cluster assumptions and help validate whether the segmentation is stable and consistent beyond K-Means.

---

### 4) Find Best K

#### 4.1 Elbow (Insertia) + Silhoutte

In [ ]:
ks = range(2, 11)
inertias = []
sil_scores = []

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_train)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_train, km.labels_))

plt.figure()
plt.plot(list(ks), inertias, marker="o")
plt.title("Elbow Method (Inertia)")
plt.xlabel("k")
plt.ylabel("Inertia")
plt.show()

plt.figure()
plt.plot(list(ks), sil_scores, marker="o")
plt.title("Silhouette Score vs k")
plt.xlabel("k")
plt.ylabel("Silhouette Score")
plt.show()

**Elbow Method (Inertia)** 

The elbow plot shows a sharp decrease in inertia from k = 2 to k = 4, after which the rate of reduction begins to slow noticealy.
Beyond k = 5-6, adding more clusters results in diminishing improvements, indicating an elbow around this region.

**Silhouette Score**

The silhouette score increases steadily from k = 2 and reaches its highest value at k = 6.
After k = 6, the silhoutte score declines, suggesting reduced cluster separation for larger k values.

**Final Choice of k**

Based on both metrics, k = 6 is selected as the optimal number of clusters.
This value provides a good balance between compactness (elbow methos), cluster separation (silhouette score), and business interpretability by producing a manageable number of distinct customer segments.

---
### 5) Evaluate Clusters

Metrics to use:

1) Silhouette (higher better)
   - Measures how similar a point is to its own cluster vs other clusters
   - Range: -1 to 1
   - Higher = better-defined clusters
     
2) Davies-Bouldin (lower better)
   - Measures cluster overlap
   - Lower value = clusters are more distinct
     
3) Calinski-Harabasz (higher better)
   - Measures ratio of between-cluster separation to within-cluster compactness
   - Higher = better clustering structure

#### Helper function

In [ ]:
def cluster_metrics(X, labels):
    return {
        "Silhouette": silhouette_score(X, labels),
        "Davies_Bouldin": davies_bouldin_score(X, labels),
        "Calinski_Harabasz": calinski_harabasz_score(X, labels)
    }

---
### 6 Train and evaluate models (k = 6)

In [ ]:
best_k = 6

#### 6.1 K-Means

In [ ]:
km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
km_labels = km.fit_predict(X_train)

kmeans_metrics = cluster_metrics(X_train, km_labels)
kmeans_metrics

#### 6.2 Agglomerative (Ward)

In [ ]:
agg = AgglomerativeClustering(n_clusters=best_k, linkage="ward")
agg_labels = agg.fit_predict(X_train)

agg_metrics = cluster_metrics(X_train, agg_labels)
agg_metrics

#### 6.3 GMM

In [ ]:
gmm = GaussianMixture(n_components=best_k, random_state=42)
gmm_labels = gmm.fit_predict(X_train)

gmm_metrics = cluster_metrics(X_train, gmm_labels)
gmm_metrics

#### 6.4 Comparison table

In [ ]:
metrics_df = pd.DataFrame({
    "KMeans": kmeans_metrics,
    "Agglomerative": agg_metrics,
    "GMM": gmm_metrics
})

metrics_df

---
**Cluster Evaluation**

Since unsupervised learning has no ground truth labels, cluster quality is evaluated using silhouette score, Davies–Bouldin index, and Calinski–Harabasz index.

K-Means and Agglomerative clustering achieve very similar silhouette scores, indicating comparable cluster separation. 
However, K-Means produces the highest Calinski–Harabasz score, suggesting stronger between-cluster separation relative to within-cluster compactness.

Gaussian Mixture Model shows a noticeably lower silhouette score and higher Davies–Bouldin index, indicating greater overlap between clusters.

---
**Final Model Selection**

Based on the evaluation metrics, **K-Means** is selected as the final clustering model.
It provides the best overall balance of compactness, separation, and interpretability, making it suitable for customer segmentation and business analysis.

---

### 7) Interpret Clusters (Characteristics)

#### 7.1 Assign clusters back to full dataset

In [ ]:
final_model = KMeans(n_clusters=best_k, random_state=42, n_init=10)
final_labels = final_model.fit_predict(X_scaled)

# K-Means numeric IDs are arbitrary; align them to the published segment profiles.
from scipy.optimize import linear_sum_assignment
reference_means = np.array([
    [25.00, 25.26, 77.61], [56.33, 54.27, 49.07],
    [41.94, 88.94, 16.97], [26.79, 57.10, 48.13],
    [32.69, 86.54, 82.13], [45.52, 26.29, 19.38]
])
raw_means = np.array([
    df.loc[final_labels == i, ["Age", "Income (k$)", "How Much They Spend"]].mean().values
    for i in range(best_k)
])
row_ids, profile_ids = linear_sum_assignment(
    ((raw_means[:, None, :] - reference_means[None, :, :]) ** 2).sum(axis=2)
)
cluster_id_map = dict(zip(row_ids, profile_ids))
final_labels = np.array([cluster_id_map[i] for i in final_labels])

df_result = df.copy()
df_result["cluster"] = final_labels
df_result.head()

#### 7.2 Cluster profile table

In [ ]:
cluster_summary = df_result.groupby("cluster").agg(
    count=("CustomerID","size"),
    male_pct=("Gender", lambda s: (s=="Male").mean()*100),
    age_mean=("Age","mean"),
    income_mean=("Income (k$)","mean"),
    spend_mean=("How Much They Spend","mean"),
    age_min=("Age","min"),
    age_max=("Age","max"),
    income_min=("Income (k$)","min"),
    income_max=("Income (k$)","max"),
    spend_min=("How Much They Spend","min"),
    spend_max=("How Much They Spend","max"),
).round(2)

cluster_summary

#### 7.3 Visualize clusters (Income vs Spending)

The Income vs Spending scatter plot visually supports the cluster profiles, showing clear separation between high-income low-spending customers, low-income high-spending customers, and premium high-spending segments.


In [ ]:
plt.figure()
plt.scatter(df_result["Income (k$)"], df_result["How Much They Spend"], c=df_result["cluster"], s=60)
plt.title("Clusters: Income vs Spending")
plt.xlabel("Income (k$)")
plt.ylabel("How Much They Spend")
plt.show()

---
### 8) Answer the 3 Business Questions

#### Q1) How to perform customer segmentation using unsupervised machine learning algorithm in Python?

Customer segmentation is performed using the following workflow:

1. Conduct exploratory data analysis (EDA) to understand distributions, relationships, and potential outliers.

2. Apply feature scaling to ensure all numeric variables contribute equally to distance-based clustering.

3. Train multiple clustering algorithms (e.g. K-Means, Agglomerative, GMM) to avoid reliance on a single model.

4. Determine the optimal number of clusters (k) using evaluation methods such as the elbow method and silhouette score.

5. Evaluate cluster structure using unsupervised metrics (silhouette score, Davies–Bouldin index, Calinski–Harabasz index).

6. Interpret clusters by analysing their demographic and behavioural characteristics.

#### Q2) What are the characteristics of each customer cluster?

The characteristics of each cluster are derived from the cluster summary table and include:
    
- Cluster size
    
- Age (range and mean)
    
- Income (mean and range)
    
- Spending behaviour (mean and range)
    
- Gender composition (used for profiling, not clustering)

These attributes allow each cluster to be interpreted as a distinct customer segment with different spending patterns and purchasing potential.
    

---

**Cluster 0: Young, Low Income, High Spenders**

- Age: young (18–35, mean ≈ 25)

- Income: low (mean ≈ 25k)

- Spending: high (mean ≈ 78)

**Interpretation:**
These customers spend heavily despite having low income, suggesting impulsive or lifestyle-driven spending.

---

**Cluster 1: Older, Mid Income, Moderate Spenders**

- Age: older (43–70, mean ≈ 56)

- Income: mid-range (mean ≈ 54k)

- Spending: moderate (mean ≈ 49)

**Interpretation:**
This group shows stable and controlled spending behaviour, likely prioritising value and necessity over impulse purchases.

---

**Cluster 2: High Income, Low Spenders**

- Age: wide range (19–59)

- Income: high (mean ≈ 89k)

- Spending: low (mean ≈ 17)

**Interpretation:**
Despite strong purchasing power, these customers spend relatively little, indicating untapped potential or price sensitivity.

---

**Cluster 3: Young Adults, Mid Income, Moderate Spenders**

- Age: young adults (18–40)

- Income: mid-range (mean ≈ 57k)

- Spending: moderate (mean ≈ 48)

**Interpretation:**
This cluster represents average customers with balanced income and spending behaviour.

---

**Cluster 4: Mid Income, Very High Spenders**

- Age: adults (27–40)

- Income: high-mid (mean ≈ 87k)

- Spending: very high (mean ≈ 82)

**Interpretation:**
These are premium customers who convert income into spending efficiently and represent strong revenue contributors.

---

**Cluster 5: Older, Low Income, Low Spenders**

- Age: older (20–67, mean ≈ 46)

- Income: low (mean ≈ 26k)

- Spending: low (mean ≈ 19)

**Interpretation:**
This group shows limited purchasing power and conservative spending, contributing less to overall revenue.

---

#### Q3) Which group of customer needs to pay more attention to?

**Primary focus: Cluster 2 (High Income, Low Spending)**

Cluster 2 requires the most attention as it consists of high-income customers with relatively low spending. 
This suggests strong untapped potential, where targeted promotions, personalised recommendations, or premium positioning could increase their spending.

**Retention focus: Cluster 4 (High Income, High Spending)**

Cluster 4 should be prioritised for retention strategies, as these customers already demonstrate high spending behaviour and contribute significantly to revenue.

**Lower priority: Cluster 5 (Low Income, Low Spending)**

Cluster 5 shows both low income and low spending and may offer limited growth potential compared to other clusters.